# Vẽ biểu đồ kết quả thực nghiệm
So sánh DQN / Double DQN / Dijkstra baseline từ các CSV trong `results/csv/`.

Chạy cell bên dưới sau khi đã huấn luyện `train_dqn.py`, `train_ddqn.py` và `baseline.py`.

In [ ]:
%matplotlib inline
import glob, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_DIR = 'results/csv'
FIG_DIR = 'results/figures'
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
def parse_tag(path):
    name = os.path.basename(path).replace('.csv', '')
    m = re.match(r'(dqn|ddqn)_load([0-9.]+)', name)
    return (m.group(1), float(m.group(2))) if m else None

training = {}
for f in glob.glob(f'{CSV_DIR}/dqn*.csv') + glob.glob(f'{CSV_DIR}/ddqn*.csv'):
    tag = parse_tag(f)
    if tag is None:
        continue
    algo, load = tag
    df = pd.read_csv(f)
    df['algo'] = algo
    training.setdefault(load, []).append((algo, df))

baseline = None
if os.path.exists(f'{CSV_DIR}/baseline.csv'):
    baseline = pd.read_csv(f'{CSV_DIR}/baseline.csv')
print('loads:', sorted(training.keys()))

In [ ]:
colors = {'dqn': '#1f77b4', 'ddqn': '#ff7f0e'}
for load, entries in training.items():
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for algo, df in entries:
        c = colors.get(algo, 'black')
        axes[0, 0].plot(df['episode'], df['train_reward'].rolling(20, min_periods=1).mean(), label=algo, color=c)
        axes[0, 1].plot(df['episode'], df['train_loss'].rolling(20, min_periods=1).mean(), label=algo, color=c)
        axes[1, 0].plot(df['episode'], df['train_avg_delay_ms'].rolling(20, min_periods=1).mean(), label=algo, color=c)
        axes[1, 1].plot(df['episode'], df['train_loss_rate'].rolling(20, min_periods=1).mean(), label=algo, color=c)
        ev = df.dropna(subset=['eval_reward'])
        if len(ev):
            axes[0, 0].plot(ev['episode'], ev['eval_reward'], 'o', color=c, alpha=0.5, label=f'{algo} (eval)')
    axes[0, 0].set_title('Reward/episode'); axes[0, 0].legend()
    axes[0, 1].set_title('Loss'); axes[0, 1].legend()
    axes[1, 0].set_title('Avg delay (ms)'); axes[1, 0].legend()
    axes[1, 1].set_title('Packet loss rate'); axes[1, 1].legend()
    for ax in axes.flat:
        ax.set_xlabel('episode')
    fig.suptitle(f'Training curves (load={load})')
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/training_load{load}.png', dpi=150)
    plt.show()

In [ ]:
rows = []
for load, entries in training.items():
    for algo, df in entries:
        ev = df.dropna(subset=['eval_reward'])
        if len(ev) == 0:
            continue
        last = ev.tail(1).iloc[0]
        rows.append({'load': load, 'algo': algo, 'reward': last['eval_reward'],
                     'delay_ms': last['eval_avg_delay_ms'],
                     'loss_rate': last['eval_loss_rate'],
                     'throughput': last['eval_throughput']})
if baseline is not None:
    for _, r in baseline.iterrows():
        rows.append({'load': r['load'], 'algo': 'dijkstra', 'reward': r['episode_reward'],
                     'delay_ms': r['avg_delay_ms'], 'loss_rate': r['packet_loss_rate'],
                     'throughput': r['throughput']})
summary = pd.DataFrame(rows)
summary.to_csv(f'{CSV_DIR}/summary.csv', index=False)
summary

In [ ]:
metrics = ['reward', 'delay_ms', 'loss_rate', 'throughput']
for load in sorted(summary['load'].unique()):
    sub = summary[summary['load'] == load]
    algos = sorted(sub['algo'].unique())
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, metric in zip(axes, metrics):
        for i, algo in enumerate(algos):
            row = sub[sub['algo'] == algo].iloc[0]
            ax.bar(i, row[metric], width=0.5, label=algo)
        ax.set_title(metric)
        ax.set_xticks(range(len(algos)))
        ax.set_xticklabels(algos)
        ax.legend()
    fig.suptitle(f'Final comparison (load={load})')
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/comparison_load{load}.png', dpi=150)
    plt.show()